In [ ]:
import os
import numpy as np
import scipy
import matplotlib.pyplot as plt

from scipy.sparse import lil_matrix, csr_matrix, csc_matrix, coo_matrix, bsr_matrix
from scipy.optimize import nnls, least_squares
from scipy.linalg import circulant


from util.Processing import discretize, td_nnlsr_deconvolve

In [ ]:
mV_per_ADC = 1000. / 4096.
bits = 8
tstep = 12e-9  # Sampling rate in seconds (40 MHz)

In [ ]:
for file in os.listdir('.'):
    if 'txt' not in file:
        continue
        
    real_trace = np.loadtxt(file, skiprows=1) 
    time = real_trace[:, 0]  # Relative time in µs
    adc = real_trace[:, 1]  # ADC amplitudes
    
    plt.figure(figsize=(5,3))
    plt.plot(time*1E6, adc * mV_per_ADC)
    plt.xlabel('Microseconds')
    plt.ylabel('mV')
    plt.title(file)
    # plt.ylim(0, 250)
    

In [ ]:
pulse = np.loadtxt('NaI_trace_pulse_220726_045157_buffer_0.txt', skiprows=1)[:, 1]
mode = pulse[0]
index = np.arange(pulse.size)
start = index[pulse != mode][0] - 1 # one before first non-mode
end = index[pulse != mode][-1] + 1 # one after last non-mode
pulse = pulse[start:end+1]
print(start, end)

pulse /= np.max(pulse)

plt.plot(pulse)

In [ ]:
# Prepare the real trace for NNLSR
real_trace = np.loadtxt('NaI_trace_filtered_220726_045157_buffer_0.txt', skiprows=1) 
time = real_trace[:, 0]  # Relative time in µs
adc = real_trace[:, 1]  # ADC amplitudes

In [ ]:
# Window
start = 12800
end = 13500
# start = 0
# end = adc.size
deconvolve = True

trace = adc.copy()[start:end]
index = np.arange(start, end)

base_est, _ = scipy.stats.mode(trace, keepdims=True)
trace -= base_est
trace *= mV_per_ADC


plt.figure(figsize=(14, 14), dpi=200)
plt.plot(index, trace, label='THOR Trace')

if deconvolve:
    nnlsr_deconv = td_nnlsr_deconvolve(trace, pulse)  
    
    nnlsr_gt0 = nnlsr_deconv > 0
    plt.plot(index[nnlsr_gt0], nnlsr_deconv[nnlsr_gt0], color='red', marker='.', markersize=1, linestyle='', label='Straight NNLSR')
    
    section_index = np.arange(nnlsr_deconv.size)
    index_gt0 = section_index[nnlsr_gt0]
    diff_nonzero = np.diff(index_gt0)
    print(diff_nonzero)
        
    groups = []
    group = diff_nonzero[0] == 1
    print(group)
    group_start = 0
    for i, diff in enumerate(diff_nonzero):
        if i == diff_nonzero.size-1:
            if group:
                groups.append(index_gt0[np.arange(group_start, i+1)])
        elif group:
            if diff_nonzero[i] != 1:
                groups.append(index_gt0[np.arange(group_start, i+1)])
                group = False
        else:
            if diff_nonzero[i] == 1:
                group_start = i
                group = True
                
    print(groups)
    summed_index = []
    summed = []
    for group in groups:  
        # print(group)
        # print(section_index[group], np.sum(trace[section_index[group]]))
        # print()
        
        summed_index.append(section_index[group][0] + index[0])       
        summed.append(np.sum(trace[section_index[group]]))
        
    plt.plot(summed_index, summed, color='green', marker='.', markersize=6, linestyle='', label='Contiguous Summed')
    
plt.legend()
    
    

In [ ]:
# Window
start = 12800
end = 13500
# start = 20000
# end = 23000
# start = 0
# end = adc.size
deconvolve = True

trace = adc.copy()[start:end]
index = np.arange(start, end)

base_est, _ = scipy.stats.mode(trace, keepdims=True)
trace -= base_est
trace *= mV_per_ADC


plt.figure(figsize=(10, 10), dpi=200)
plt.plot(index, trace, label='THOR Trace')

if deconvolve:
    def nnlsr_residual(x, C, trace):
        out = trace - C @ x
        return out
    
    c = np.concatenate((pulse, np.zeros(trace.size - pulse.size)), axis=0)
    circ = circulant(c)
    # C_jac = csc_matrix(lil_matrix(C)) # note that this shape should match the sparsity operations so no transpose...
    sparsity = csc_matrix(lil_matrix((circ != 0)))
    C = csc_matrix(lil_matrix(circ))
    x0 = np.ones(trace.size)
    
    solution = least_squares(fun=nnlsr_residual,
                             jac='cs',
                             method='trf', # use with bounds!
                             bounds=[0, np.inf],
                             args=(C, trace),
                             x0=x0,
                             x_scale=10,
                             diff_step=1,
                             loss = 'linear',
                             jac_sparsity=C, # Speeds things up significantly with sparse matrices
                             ftol=1E-15,
                             max_nfev=100,
                             verbose=2
                             )
    deconv = solution.x
    
    nnlsr_gt0 = nnlsr_deconv > 0
    plt.plot(index[nnlsr_gt0], nnlsr_deconv[nnlsr_gt0], color='red', marker='.', markersize=1, linestyle='', label='Straight NNLSR')
    
    # section_index = np.arange(nnlsr_deconv.size)
    # index_gt0 = section_index[nnlsr_gt0]
    # diff_nonzero = np.diff(index_gt0)
    # print(diff_nonzero)
    #     
    # groups = []
    # group = diff_nonzero[0] == 1
    # print(group)
    # group_start = 0
    # for i, diff in enumerate(diff_nonzero):
    #     if i == diff_nonzero.size-1:
    #         if group:
    #             groups.append(index_gt0[np.arange(group_start, i+1)])
    #     elif group:
    #         if diff_nonzero[i] != 1:
    #             groups.append(index_gt0[np.arange(group_start, i+1)])
    #             group = False
    #     else:
    #         if diff_nonzero[i] == 1:
    #             group_start = i
    #             group = True
    #             
    # print(groups)
    # summed_index = []
    # summed = []
    # for group in groups:  
    #     print(group)
    #     print(section_index[group], np.sum(trace[section_index[group]]))
    #     print()
    #     
    #     summed_index.append(section_index[group][0] + index[0])       
    #     summed.append(np.sum(trace[section_index[group]]))
        
    # plt.plot(summed_index, summed, color='green', marker='.', markersize=6, linestyle='', label='Contiguous Summed')
        
        
plt.legend()
    